# J3-A — Classer les demandes avec Trainer

**NLP Avancé (BERT, GPT, Hugging Face) · M2 IA · Jour 3 · 140 min · Chrys NIONGOLO**

Travail en binôme : alterner la personne qui conduit et celle qui explique. Enregistrer une copie dans Drive, puis conserver le notebook exécuté et le JSON final.

## À la fin, vous saurez

- Construire une séparation qui garde les paraphrases ensemble.
- Comparer une baseline à un encodeur adapté.
- Expliquer macro-F1 et la matrice de confusion, puis exporter le modèle.

## Parcours de l’après-midi

| Étape | Minutes |
| Cadrage, groupes, baseline | 30 |
| Tokenisation, lots, entraînement | 40 |
| Validation et expérience contrôlée | 35 |
| Test, erreurs et export | 35 |

Les durées sont des budgets d’activité incluant lecture, discussion, essais et débrief ; ce ne sont pas des durées de calcul promises. Les corpus intégrés sont originaux, fictifs et minuscules. Ils servent à comprendre un protocole, pas à certifier une qualité métier.

**Règle d’expérience :** écrire une prédiction avant de lancer une variante ; changer un seul facteur ; choisir sur la validation ; ouvrir le test après ce choix.


## 0. Préparer Colab

Importer ce fichier dans https://colab.research.google.com/ via Fichier → Importer.
Choisir Exécution → Modifier le type d’exécution → Python 3 → GPU T4 si disponible.
La disponibilité du T4 dépend des quotas Colab. Aucun compte Hugging Face n’est requis
pour les modèles publics utilisés ici. Une connexion internet est nécessaire pour les
bibliothèques et les poids ; les données pédagogiques sont déjà dans le notebook.

Exécuter la cellule suivante **avant les autres imports**. Si elle demande un redémarrage,
choisir Exécution → Redémarrer la session puis relancer depuis le début. Ne pas réinstaller
un autre PyTorch dans un runtime CUDA fonctionnel. Les versions sont volontairement fixées.
Aucun entraînement n’est lancé par cette cellule. Aucun envoi au Hub n’est automatique.

**Runpod L4 facultatif :** ce notebook fonctionne dans Jupyter avec Python 3.10+
et PyTorch CUDA préinstallé. Les chemins sont relatifs ; aucun import Google Colab
n’est requis. Arrêter le Pod après sauvegarde des résultats pour interrompre sa facturation.


In [ ]:
PINNED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.6', 'scikit-learn': '1.7.2', 'transformers': '4.56.2', 'tokenizers': '0.22.0', 'huggingface-hub': '0.35.3', 'accelerate': '1.10.1', 'datasets': '4.1.1'}
import sys, subprocess, importlib.metadata as metadata
assert sys.version_info >= (3, 10), "Utiliser Python 3.10 ou ultérieur."
module_names = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",
                "huggingface-hub": "huggingface_hub", "rouge-score": "rouge_score"}
to_install = []
loaded_mismatch = []
for package, version in PINNED.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", version) != version:
        loaded_mismatch.append(package)
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        to_install.append(f"{package}=={version}")
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
if loaded_mismatch:
    raise RuntimeError("Redémarrer la session Colab, puis relancer depuis le début : "
                       + ", ".join(loaded_mismatch))
print("Versions de référence installées. Continuer avec la cellule suivante.")


In [ ]:
import os, json, time, random, platform, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
versions = {p: metadata.version(p) for p in PINNED}
versions["python"] = platform.python_version()
results = {"seed": SEED, "versions": versions, "validation": "exécuté par le binôme"}
OUTPUT = Path("resultats")
OUTPUT.mkdir(exist_ok=True)
display(pd.Series(versions, name="version effective"))


In [ ]:
import torch
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
versions["torch"] = torch.__version__
results["device"] = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print("Périphérique :", results["device"])
if DEVICE == "cuda":
    print("VRAM totale (Gio) :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print("Pas de GPU : les sections indiquées restent utilisables ; ne pas présenter un modèle non entraîné comme adapté.")


## 1. Une tâche, cinq intentions

Affecter un ticket à livraison, facturation, compte, retour ou technique.
La règle d’annotation décrit **l’action demandée**, pas un mot isolé : une page de facture blanche
peut relever d’un problème technique. Discuter ce cas avant d’entraîner.

Nous disposons de 80 textes : 40 scénarios avec deux formulations chacun. Les deux variantes
restent dans la même partition. Les scénarios 0–4 servent à apprendre, le 5 à sélectionner,
les 6–7 au test. Une séparation aléatoire ligne par ligne ferait fuir des paraphrases.
Le corpus reste petit, écrit dans un style commun et ne mesure pas la diversité de clients réels.


In [ ]:
SUPPORT = {'livraison': [('Le suivi de mon colis ne bouge plus depuis lundi.', 'Mon paquet semble bloqué au centre de tri.'), ('Le livreur indique livré mais ma boîte est vide.', "Je n'ai rien reçu alors que le suivi annonce une remise."), ('Puis-je choisir un autre point relais ?', 'Je voudrais faire déposer ma commande dans un relais différent.'), ("L'adresse sur mon envoi est incomplète.", 'Il manque le numéro de rue pour la livraison.'), ('Le paquet est reparti chez vous avant mon passage.', "Je n'ai pas pu récupérer le colis à temps au relais."), ('Les deux articles voyagent-ils dans le même carton ?', "Je n'ai qu'un numéro de suivi pour plusieurs produits."), ('Le transporteur a reporté son passage sans date.', "Aucune nouvelle tentative de livraison n'est annoncée."), ('Mon colis doit arriver au bureau fermé samedi.', 'La réception de mon entreprise sera absente lors du passage.')], 'facturation': [("J'ai été débité deux fois pour la même commande.", 'Deux paiements identiques apparaissent sur mon relevé.'), ('Où récupérer la facture de mon achat ?', "J'ai besoin d'un justificatif comptable pour la commande."), ("La remise affichée n'a pas été appliquée au paiement.", 'Le total payé ignore le code promotionnel saisi.'), ('Le paiement est refusé alors que ma carte fonctionne ailleurs.', "Votre page n'accepte pas ma carte bancaire."), ('Le nom de mon entreprise manque sur la facture.', 'Pouvez-vous corriger les informations de facturation ?'), ("Une somme reste en attente après l'annulation.", "L'autorisation bancaire semble encore bloquée."), ('Le reçu est dans une devise différente de mon panier.', 'Je ne comprends pas la conversion sur mon justificatif.'), ('La taxe indiquée sur le document paraît incorrecte.', 'Le montant de TVA de ma facture mérite une vérification.')], 'compte': [("J'ai oublié mon mot de passe.", 'Je ne me souviens plus du code pour accéder à mon espace.'), ("Le courriel de réinitialisation n'arrive pas.", "Le lien pour retrouver mon accès n'est pas dans ma boîte mail."), ("Je voudrais changer l'adresse email de mon profil.", 'Mon espace utilise encore mon ancien courriel.'), ('Comment fermer définitivement mon compte ?', 'Je souhaite supprimer mon profil client.'), ('Le code de connexion à deux facteurs est expiré.', 'La double authentification bloque mon accès.'), ('Je ne reconnais pas une connexion récente.', 'Mon historique montre un accès qui ne vient pas de moi.'), ('Je veux récupérer les données de mon profil.', 'Comment obtenir une copie des informations de mon compte ?'), ('Mon adresse mail semble associée à deux profils.', 'Je retrouve deux espaces clients pour la même identité.')], 'retour': [("La taille reçue ne me convient pas, je veux renvoyer l'article.", 'Je souhaite retourner ce vêtement trop petit.'), ('Le produit arrivé cassé doit vous être retourné.', 'Je voudrais organiser le renvoi de cet article abîmé.'), ("Où trouver l'étiquette de retour ?", "Je n'arrive pas à télécharger le bordereau de renvoi."), ('Mon retour a été reçu, où en est son traitement ?', "Avez-vous vérifié le produit que j'ai renvoyé ?"), ("Je préfère échanger l'article plutôt que le garder.", 'Est-il possible de remplacer ce produit par une autre taille ?'), ("J'ai jeté l'emballage, puis-je tout de même renvoyer ?", "Le carton d'origine est perdu pour mon retour."), ("Je souhaite renvoyer seulement une partie de l'achat.", "Comment retourner un seul article d'une commande multiple ?"), ('Le transporteur refuse mon paquet de retour trop volumineux.', 'Le renvoi ne rentre pas dans le format accepté au relais.')], 'technique': [("L'application se ferme quand j'ouvre mon panier.", 'Votre appli plante au moment de voir les articles ajoutés.'), ('La page produit reste blanche sur mon navigateur.', 'Les fiches articles ne se chargent plus sur le site.'), ('Le bouton de recherche ne fait rien.', 'Je saisis un mot mais la recherche du site ne démarre pas.'), ("L'image du produit ne s'affiche pas.", 'Toutes les photos sont remplacées par une icône cassée.'), ('La version mobile masque le bouton continuer.', "Sur mon téléphone je ne peux pas passer à l'étape suivante."), ("Le site boucle lorsqu'on accepte les cookies.", 'La fenêtre de consentement réapparaît sans cesse.'), ('Votre formulaire refuse les caractères accentués.', "Le champ de saisie signale une erreur quand j'écris é."), ("La page est inutilisable avec mon lecteur d'écran.", "La navigation au clavier n'atteint pas les boutons du site.")]}
LABELS = sorted(SUPPORT)
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}
rows = []
for label, groups in SUPPORT.items():
    for g, pair in enumerate(groups):
        for variant, text in enumerate(pair):
            rows.append({"text": text, "label": label2id[label], "category": label,
                         "group": f"{label}_{g}", "scenario": g,
                         "split": "train" if g < 5 else "validation" if g == 5 else "test"})
frame = pd.DataFrame(rows)
assert frame.text.is_unique
train_df = frame[frame.split == "train"].reset_index(drop=True)
val_df = frame[frame.split == "validation"].reset_index(drop=True)
test_df = frame[frame.split == "test"].reset_index(drop=True)
for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert set(a.group).isdisjoint(b.group), "Fuite de paraphrases entre partitions."
display(pd.crosstab(frame.category, frame.split))
results["data"] = {"source": "Corpus original fictif CYBERSUP, 2026-10-04",
                   "rows": len(frame), "split_unit": "scénario/paraphrases",
                   "sha256": hashlib.sha256(frame.to_json().encode()).hexdigest()}


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, ConfusionMatrixDisplay
baseline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), strip_accents="unicode"),
                         LogisticRegression(max_iter=1000, C=2.0, random_state=SEED))
baseline.fit(train_df.text, train_df.label)
val_pred_baseline = baseline.predict(val_df.text)
baseline_val_f1 = f1_score(val_df.label, val_pred_baseline, labels=list(id2label), average="macro", zero_division=0)
print("Macro-F1 validation baseline :", round(baseline_val_f1, 3))
results["baseline_validation_macro_f1"] = float(baseline_val_f1)


In [ ]:
# Montrer le risque d'une séparation naïve, sans l'utiliser pour entraîner.
from sklearn.model_selection import train_test_split
a, b = train_test_split(frame, test_size=0.25, random_state=SEED, stratify=frame.label)
overlap = set(a.group) & set(b.group)
print("Scénarios partagés par la séparation naïve :", len(overlap))
display(train_df[["text", "category", "group"]].head(8))
reponses = {"unite_split": "À compléter", "cas_ambigu": "À compléter", "hypothese_gain": "À compléter"}


### Repères formateur — éléments de correction

Un score artificiellement haut peut venir du partage d’un scénario et non du nombre exact de mots communs. Une partition par client, incident ou période serait souvent plus adaptée aux vrais tickets. La baseline est entraînée uniquement sur train. Les dix exemples de validation donnent une estimation très instable ; ne pas lancer une grande recherche d’hyperparamètres dessus.


## 2. Brancher les quatre pièces

Le tokenizer produit des identifiants et un masque de padding. Le collator complète les séquences
à la longueur du plus long texte du lot. Le modèle encode puis classe. Trainer gère les lots,
la loss, les gradients, l’évaluation et les sauvegardes.

**Avant exécution (5 min).** Estimer la forme des logits pour un lot de huit textes et cinq labels.
Le modèle de base ne possède pas de tête entraînée pour nos cinq catégories : un avertissement
de poids nouvellement initialisés est normal. Le T4 utilise fp16 ; bf16 est explicitement désactivé.


In [ ]:
MODEL_ID = 'distilbert/distilbert-base-multilingual-cased'
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
MAX_LENGTH = 128
def make_dataset(table):
    ds = Dataset.from_dict({"text": table.text.tolist(), "labels": table.label.tolist()})
    return ds.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["text"])
train_ds, val_ds = make_dataset(train_df), make_dataset(val_df)
token_lengths = [len(tokenizer(text)["input_ids"]) for text in train_df.text]
print("Longueur max avant troncature :", max(token_lengths), "; limite choisie :", MAX_LENGTH)
assert max(token_lengths) <= MAX_LENGTH, "Réexaminer la troncature avant d'entraîner."
def compute_metrics(prediction):
    pred = np.argmax(prediction.predictions, axis=-1)
    return {"accuracy": accuracy_score(prediction.label_ids, pred),
            "macro_f1": f1_score(prediction.label_ids, pred, labels=list(id2label), average="macro", zero_division=0)}
RUN_TRAINING = DEVICE == "cuda"  # Sur CPU, activer volontairement après accord sur le temps disponible.
EPOCHS = 3
LEARNING_RATE = 3e-5
trainer = None
trained_model = None


In [ ]:
if RUN_TRAINING:
    set_seed(SEED)
    trained_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    # La tête de classification neuve est aléatoire : l'avertissement de poids initialisés est attendu.
    args = TrainingArguments(
        output_dir="checkpoints_classification", learning_rate=LEARNING_RATE,
        per_device_train_batch_size=8, per_device_eval_batch_size=8,
        num_train_epochs=EPOCHS, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=DEVICE == "cuda", bf16=False, report_to="none", seed=SEED,
        data_seed=SEED, logging_steps=5, dataloader_num_workers=0,
        push_to_hub=False, save_safetensors=True)
    trainer = Trainer(model=trained_model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
                      compute_metrics=compute_metrics)
    start = time.perf_counter()
    trainer.train()
    results["classification_train_seconds"] = time.perf_counter() - start
    results["classification_validation"] = trainer.evaluate()
    results["classification_config"] = {"epochs": EPOCHS, "learning_rate": LEARNING_RATE,
                                         "max_length": MAX_LENGTH, "model": MODEL_ID}
    print(results["classification_validation"])
elif trained_model is None:
    print("Entraînement non exécuté. Baseline et analyse du protocole restent disponibles.")
def predict_finetuned(texts):
    if trained_model is None:
        return None
    trained_model.eval()
    output = []
    for start in range(0, len(texts), 8):
        batch = tokenizer(list(texts[start:start + 8]), padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            logits = trained_model(**batch).logits
        output.extend(logits.argmax(-1).cpu().tolist())
    return np.array(output)


In [ ]:
collator = DataCollatorWithPadding(tokenizer)
example_batch = collator([train_ds[0], train_ds[1]])
print({key: tuple(value.shape) for key, value in example_batch.items()})
assert example_batch["input_ids"].shape == example_batch["attention_mask"].shape
assert example_batch["labels"].shape == (2,)
if trainer is not None:
    history = pd.DataFrame(trainer.state.log_history)
    display(history)
    if "eval_macro_f1" in history:
        history.dropna(subset=["eval_macro_f1"]).plot(x="epoch", y="eval_macro_f1", marker="o")
        plt.ylim(0, 1); plt.title("Validation : petit échantillon, variations possibles"); plt.show()


## 3. Une expérience contrôlée

Si le temps le permet, choisir **une seule** variante : longueur 64 contre 128 ; learning rate 1e-5 contre 3e-5 ;
une époque contre trois. Repartir du modèle de base pour chaque essai, garder la même partition,
enregistrer les valeurs et le temps. Relancer les cellules de préparation du modèle et d’entraînement,
puis sélectionner sur la validation. Ne pas exécuter la cellule de test avant le choix final.

Une loss plus faible sur train n’implique pas une meilleure généralisation. Si la baseline gagne,
expliquer son intérêt en coût et simplicité. Ne pas ajouter discrètement les exemples du test à train.


In [ ]:
variant_log = []  # Ajouter une ligne par essai : configuration, macro-F1 validation, temps.
if trainer is not None:
    variant_log.append({**results["classification_config"],
                        "validation_macro_f1": results["classification_validation"]["eval_macro_f1"],
                        "seconds": results["classification_train_seconds"]})
display(pd.DataFrame(variant_log))
choix_final = "À compléter avant le test : configuration et justification."


## 4. Test final et erreurs

Le protocole et le modèle sont maintenant figés. Évaluer baseline et modèle adapté sur les mêmes
vingt textes. Macro-F1 donne le même poids à chaque classe ; l’accuracy compte les textes corrects.
Lire aussi les erreurs : manque de données, ambiguïté d’annotation, vocabulaire nouveau ou confusion métier.


In [ ]:
test_pred_baseline = baseline.predict(test_df.text)
test_pred_ft = predict_finetuned(test_df.text.tolist())
test_rows = []
predictions = {"tfidf_logreg": test_pred_baseline}
if test_pred_ft is not None: predictions["encodeur_finetune"] = test_pred_ft
for method, pred in predictions.items():
    test_rows.append({"methode": method,
                      "macro_f1": f1_score(test_df.label, pred, labels=list(id2label), average="macro", zero_division=0),
                      "accuracy": accuracy_score(test_df.label, pred), "n": len(test_df)})
    print(method)
    print(classification_report(test_df.label, pred, labels=list(id2label), target_names=LABELS, zero_division=0))
display(pd.DataFrame(test_rows))
chosen_predictions = test_pred_ft if test_pred_ft is not None else test_pred_baseline
ConfusionMatrixDisplay.from_predictions(test_df.label, chosen_predictions, labels=list(id2label),
                                        display_labels=LABELS, xticks_rotation=35)
plt.show()
errors = test_df.assign(prediction=[id2label[int(x)] for x in chosen_predictions])
display(errors[errors.category != errors.prediction][["text", "category", "prediction", "group"]])
results.update({"test": test_rows, "choix_final": choix_final, "reponses": reponses,
                "predictions": errors.to_dict(orient="records")})


### Repères formateur — éléments de correction

Ne prescrire aucun score minimum sur ce micro-corpus. Demander au moins deux erreurs ou, s’il n’y en a pas, deux nouveaux contre-exemples formulés après le test et explicitement hors score initial. L’amélioration sur une seule graine n’établit pas une supériorité générale. Les logits ont une forme batch × 5 ; le masque de padding n’est pas le masque causal.


## 5. Sauvegarder et vérifier le rechargement

Le ZIP du modèle servira au J5. Il contient des poids, pas simplement un notebook.
Télécharger ce fichier avant de fermer Colab. Le rechargement ci-dessous vérifie que l’export
donne les mêmes prédictions sur trois textes de validation. Ce contrôle ne mesure pas la qualité métier.


In [ ]:
import shutil
if trained_model is not None:
    model_dir = Path("modele_classification")
    trained_model.save_pretrained(model_dir, safe_serialization=True)
    tokenizer.save_pretrained(model_dir)
    (model_dir / "provenance.json").write_text(json.dumps({"data": results["data"], "model": MODEL_ID,
        "train_groups": sorted(train_df.group.unique().tolist()), "labels": label2id,
        "versions": versions}, ensure_ascii=False, indent=2), encoding="utf-8")
    reloaded = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE).eval()
    batch = tokenizer(val_df.text.tolist()[:3], padding=True, truncation=True, max_length=MAX_LENGTH,
                      return_tensors="pt").to(DEVICE)
    with torch.inference_mode(): check = reloaded(**batch).logits.argmax(-1).cpu().numpy()
    assert np.array_equal(check, predict_finetuned(val_df.text.tolist()[:3]))
    del reloaded
    if DEVICE == "cuda": torch.cuda.empty_cache()
    archive = shutil.make_archive("modele_classification", "zip", root_dir=model_dir)
    print("À télécharger pour le J5 :", archive)
    results["export_reloaded"] = True
else:
    results["export_reloaded"] = False
    print("Aucun poids adapté à exporter : entraînement non exécuté.")
conclusion_binome = "À compléter : comparaison réelle, erreurs, coût, limite de validité."


## 6. Un vrai corpus français : Allociné — 40 min

Appliquer le même protocole à des critiques de films collectées. Le corpus tblard/allocine
fournit des partitions officielles de 160 000 textes train, 20 000 validation et 20 000 test.
Sa carte indique le français, une licence MIT et les labels 0=neg, 1=pos. Lire sa provenance.

Nous conservons les partitions officielles et tirons **1 000 / 200 / 200** exemples avec une graine
fixe. C’est une tâche de sentiment distincte des cinq intentions de support. Les scores ne se mélangent
pas. Nous contrôlons les doublons exacts ; sans identifiants de film/auteur, nous ne pouvons pas prouver
une indépendance par film ou personne.

Prédire l’effet de l’ironie et des critiques mitigées. Mesurer la proportion tronquée à 128 tokens.
Comparer TF-IDF au même encodeur adapté, puis conserver une erreur de négation et une annotation
discutable. Le modèle exporté reste séparé du classifieur de support utilisé par défaut au J5.

Le téléchargement est actif par défaut. En cas d’indisponibilité réseau, ce parcours sera déclaré
non exécuté ; les scores fictifs ne seront jamais présentés comme des scores Allociné. Le cache HF
peut être préparé en exécutant la cellule avant la séance.


In [ ]:
from datasets import load_dataset
ALLOCINE_ID = "tblard/allocine"
RUN_ALLOCINE = True
ALLOCINE_SIZES = {"train": 1000, "validation": 200, "test": 200}
allocine = None
allocine_metrics = []
if RUN_ALLOCINE:
    try:
        official = load_dataset(ALLOCINE_ID)
        allocine = {split: official[split].shuffle(seed=SEED).select(range(n))
                    for split, n in ALLOCINE_SIZES.items()}
        assert official["train"].features["label"].names == ["neg", "pos"]
        fingerprints = {}
        for split, ds in allocine.items():
            hashes = [hashlib.sha256(" ".join(t.casefold().split()).encode()).hexdigest() for t in ds["review"]]
            assert len(hashes) == len(set(hashes)), f"Doublons dans {split} : auditer avant entraînement."
            fingerprints[split] = set(hashes)
        for a, c in [("train", "validation"), ("train", "test"), ("validation", "test")]:
            assert fingerprints[a].isdisjoint(fingerprints[c]), "Doublons entre partitions : auditer la fuite."
        print({split: {"n": len(ds), "labels": dict(pd.Series(ds["label"]).value_counts())}
               for split, ds in allocine.items()})
        display(pd.DataFrame(allocine["train"].select(range(3))))
    except (OSError, ConnectionError) as error:
        allocine = None
        print("Allociné non exécuté : téléchargement indisponible.", type(error).__name__)
else:
    print("Parcours Allociné explicitement désactivé.")


In [ ]:
allocine_model = None
allocine_trainer = None
if allocine is not None:
    allocine_baseline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), max_features=30000, strip_accents="unicode"),
                                     LogisticRegression(max_iter=1000, C=2.0, random_state=SEED))
    allocine_baseline.fit(allocine["train"]["review"], allocine["train"]["label"])
    val_pred = allocine_baseline.predict(allocine["validation"]["review"])
    print("Allociné baseline macro-F1 validation :", f1_score(allocine["validation"]["label"], val_pred, average="macro"))
    allocine_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    lengths = [len(allocine_tokenizer(t)["input_ids"]) for t in allocine["train"]["review"]]
    print("Part tronquée à 128 tokens :", float(np.mean(np.array(lengths) > 128)))
    def tokenize_allocine(batch):
        output = allocine_tokenizer(batch["review"], truncation=True, max_length=128)
        output["labels"] = batch["label"]
        return output
    allocine_ds = {split: ds.map(tokenize_allocine, batched=True, remove_columns=ds.column_names)
                   for split, ds in allocine.items()}
    def allocine_compute_metrics(prediction):
        pred = prediction.predictions.argmax(-1)
        return {"macro_f1": f1_score(prediction.label_ids, pred, labels=[0, 1], average="macro", zero_division=0),
                "accuracy": accuracy_score(prediction.label_ids, pred)}
    if DEVICE == "cuda":
        if trained_model is not None: trained_model.to("cpu")
        if trainer is not None:
            trainer.optimizer = None
            trainer.lr_scheduler = None
        torch.cuda.empty_cache()
        set_seed(SEED)
        allocine_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2,
            id2label={0: "neg", 1: "pos"}, label2id={"neg": 0, "pos": 1})
        allocine_args = TrainingArguments(output_dir="checkpoints_allocine", num_train_epochs=2,
            learning_rate=3e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
            eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
            fp16=True, bf16=False, report_to="none", seed=SEED, data_seed=SEED,
            dataloader_num_workers=0, logging_steps=25, push_to_hub=False)
        allocine_trainer = Trainer(model=allocine_model, args=allocine_args,
            train_dataset=allocine_ds["train"], eval_dataset=allocine_ds["validation"],
            processing_class=allocine_tokenizer, data_collator=DataCollatorWithPadding(allocine_tokenizer),
            compute_metrics=allocine_compute_metrics)
        allocine_trainer.train()
        print("Allociné encodeur validation :", allocine_trainer.evaluate())
    else:
        print("Allociné : baseline exécutée, adaptation encodeur non exécutée sans GPU.")


### Test Allociné après gel de la configuration

Comparer sur les mêmes 200 critiques issues du test officiel. Ce sous-échantillon ne reproduit
pas le benchmark complet. Pour poursuivre cette tâche au J5, préparer un zero-shot pos/neg,
le même test fixé à l’avance et le modèle adapté correspondant. Le classifieur à deux labels
ne peut pas être chargé dans la démo de routage à cinq catégories.


In [ ]:
if allocine is not None:
    y_allocine = np.array(allocine["test"]["label"])
    allocine_predictions = {"tfidf_logreg": allocine_baseline.predict(allocine["test"]["review"])}
    if allocine_trainer is not None:
        allocine_predictions["encodeur_finetune"] = allocine_trainer.predict(allocine_ds["test"]).predictions.argmax(-1)
    for method, pred in allocine_predictions.items():
        allocine_metrics.append({"methode": method, "n": len(y_allocine),
            "macro_f1": f1_score(y_allocine, pred, labels=[0, 1], average="macro", zero_division=0),
            "accuracy": accuracy_score(y_allocine, pred)})
    display(pd.DataFrame(allocine_metrics))
    final_pred = allocine_predictions.get("encodeur_finetune", allocine_predictions["tfidf_logreg"])
    wrong = np.flatnonzero(final_pred != y_allocine)[:8]
    display(pd.DataFrame([{"review": allocine["test"][int(i)]["review"],
        "attendu": int(y_allocine[i]), "predit": int(final_pred[i])} for i in wrong]))
    if allocine_model is not None:
        allocine_dir = Path("modele_allocine")
        allocine_model.save_pretrained(allocine_dir, safe_serialization=True)
        allocine_tokenizer.save_pretrained(allocine_dir)
        (allocine_dir / "provenance.json").write_text(json.dumps({"dataset": ALLOCINE_ID,
            "source": "https://huggingface.co/datasets/tblard/allocine", "seed": SEED,
            "sample_sizes": ALLOCINE_SIZES, "official_splits_preserved": True,
            "metrics": allocine_metrics, "versions": versions}, ensure_ascii=False, indent=2), encoding="utf-8")
        import shutil
        print("Export distinct :", shutil.make_archive("modele_allocine", "zip", root_dir=allocine_dir))
results["allocine"] = {"executed": allocine is not None, "dataset": ALLOCINE_ID,
    "sample_sizes": ALLOCINE_SIZES, "metrics": allocine_metrics,
    "encoder_trained": allocine_model is not None, "limit": "sous-échantillon, pas benchmark complet"}


### Repères formateur — éléments de correction

Sur des critiques réelles, sarcasme, contrastes et sentiment mitigé exposent d’autres erreurs. Lire le ratio de troncature : la conclusion d’une critique peut se trouver à la fin. Les partitions officielles sont préservées, sans preuve d’indépendance par film/auteur. Comparer qualité et coût sur le sous-échantillon ; aucun score n’est garanti.


## Ressources pour prolonger

- [HF — Classification avec Trainer](https://huggingface.co/docs/transformers/v4.56.2/en/tasks/sequence_classification)
- [Carte DistilBERT multilingue](https://huggingface.co/distilbert/distilbert-base-multilingual-cased)
- [Allociné — carte, provenance, splits et licence](https://huggingface.co/datasets/tblard/allocine)

Liens et API consultés le 4 octobre 2026. Les exercices et données de ce notebook sont originaux. Les téléchargements de modèles et l’exécution Colab doivent être vérifiés avant la séance.


## Trace à rendre

Compléter vos réponses et ajouter une observation appuyée par une sortie réelle.
Exécuter la dernière cellule, télécharger le JSON dans le panneau Fichiers de Colab,
puis télécharger votre notebook avec ses sorties. Les fichiers du runtime sont temporaires.
Ne pas remplacer une mesure absente par un chiffre plausible. Indiquer les étapes non exécutées.


In [ ]:
results["notebook"] = '03_j3_classification'
results["conclusion_binome"] = globals().get("conclusion_binome", "À compléter")

result_file = OUTPUT / "j3_classification_resultats.json"
result_file.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Fichier à télécharger :", result_file.resolve())
